# Rendu TP validation
Chai Yulin, Pagano Lucas

In [24]:
#using python 3.6
import itertools
import numpy as np
import pandas as pd
import matplotlib
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

### Chargement et préparation des données

On charge les données dans un dataframe, on le mélange car on ne sait pas comment il était ordonné au préalable, et on vérifie que les classes sont équilibrées.

In [25]:
file = "wave5000.data"
data = pd.read_csv(file, header=None)
#on mélange le dataframe
data.sample(frac=1).reset_index(inplace=True, drop=True)
data[21].value_counts()

2    1696
0    1657
1    1647
Name: 21, dtype: int64

Les classes sont équilibrées, mais lors de notre tirage au sort pour échantilloner le jeu de données, on pourrait perturber cet équilibrage par malchance. Ainsi, on crée les fonctions suivantes pour assurer qu'on partitionne le jeu de données en respectant la distribution de classes.

In [26]:
def split_to_train_test(df, train_frac=0.8):
    """Fonction pour split un dataframe en test et train et garder la proportion entre classes"""
    nb_features = len(df.columns) - 1
    train_df, test_df = pd.DataFrame(), pd.DataFrame()
    labels = df[nb_features].unique()
    for lbl in labels:
        lbl_df = df[df[nb_features] == lbl]
        lbl_train_df = lbl_df.sample(frac=train_frac)
        lbl_test_df = lbl_df.drop(lbl_train_df.index)
        train_df = train_df.append(lbl_train_df)
        test_df = test_df.append(lbl_test_df)

    return train_df, test_df

def split_val_croisee(df, nb_groupes=10):
    """Fonction pour split un dataframe en k groupes et garder la proportion entre classes"""
    nb_features = len(df.columns) - 1

    df_split = {i:pd.DataFrame() for i in range(nb_groupes)}

    labels = df[nb_features].unique()
    for lbl in labels:
        lbl_df = df[df[nb_features] == lbl]
        splits = np.array_split(lbl_df, nb_groupes)
        for index, split in enumerate(splits):
            df_split[index] = df_split[index].append(split)

    return df_split


On teste que les fonctions de split fonctionnent en affichant la proportion de chaque classe dans les dataframes créés :

In [32]:
train, test = split_to_train_test(data)
print("Validation simple :")
print(train[21].value_counts()/len(train))
print(test[21].value_counts()/len(test))

print("\nValidation croisée : ")
split = split_val_croisee(data, nb_groupes=5)
for key, dataframe in split.items():
    print(dataframe[21].value_counts()/len(dataframe))

Validation simple :
2    0.339165
0    0.331417
1    0.329418
Name: 21, dtype: float64
2    0.339339
0    0.331331
1    0.329329
Name: 21, dtype: float64

Validation croisée : 
2    0.339321
0    0.331337
1    0.329341
Name: 21, dtype: float64
2    0.338661
0    0.331668
1    0.329670
Name: 21, dtype: float64
2    0.339339
0    0.331331
1    0.329329
Name: 21, dtype: float64
2    0.339339
0    0.331331
1    0.329329
Name: 21, dtype: float64
2    0.339339
0    0.331331
1    0.329329
Name: 21, dtype: float64


## Fonctions de validation

Note préliminaire : On utilise dans les fonctions de validation le taux de réussite et non le taux d'erreur en tant que mesure de précision.

### Validation répétée

In [4]:
def validation_repetee(df, nombre_repetitions, clf):

    scores = []
    nb_features = len(df.columns) - 1

    for i in range(nombre_repetitions):
        train, test = split_to_train_test(df)

        train_x = train.values[:, :nb_features]
        train_y = train.values[:, -1]
        test_x = test.values[:, :nb_features]
        test_y = test.values[:, -1]
        clf.fit(train_x, train_y)
        predict = clf.predict(test_x)
        
        #précision
        taux_reussite = sum(predict == test_y) / len(predict)
        scores.append(taux_reussite)

    average = sum(scores)/len(scores)
    std = np.std(scores)

    return average, std

clf_ = MLPClassifier()
precision, robustesse = validation_repetee(data, nombre_repetitions=10, clf=clf_)
print("Précision (taux de réussite moyen) : {}\n"
      "Robustesse (écart-type du taux de réussite) : {}".format(precision, robustesse))

Précision (taux de réussite moyen) : 0.8618618618618619
Robustesse (écart-type du taux de réussite) : 0.01198696086480297


### Leave one out
On utilise seulement les 200 premières lignes du jeu de données car la procédure est trop longue sinon.

In [13]:
def leave_one_out(df, clf):

    nb_features = len(df.columns) - 1
    scores = []
    # À chaque itération, on choisit la ligne comme ensemble de test, et le reste comme ensemble d'entraînement
    for i in range(len(df)):
        test = df.iloc[i]
        train = df.drop(i)

        train_x = train.values[:, :nb_features]
        train_y = train.values[:, -1]
        test_x = test.values[:nb_features].reshape(1,-1)
        test_y = test.values[-1]
        clf.fit(train_x, train_y)
        predict = clf.predict(test_x)
        
        # précision
        taux_reussite = sum(predict == test_y) / len(predict)
        scores.append(taux_reussite)

    average = sum(scores) / len(scores)
    std = np.std(scores)

    return average, std


clf_ = DecisionTreeClassifier()
precision, robustesse = leave_one_out(data.head(200), clf=clf_)
print("Précision (taux de réussite moyen) : {}\n"
      "Robustesse (écart-type du taux de réussite) : {}".format(precision, robustesse))

Précision (taux de réussite moyen) : 0.725
Robustesse (écart-type du taux de réussite) : 0.4465142774872938


### Validation croisée

In [9]:
def validation_croisee(df, nb_groupes, clf):

    nb_features = len(df.columns) - 1

    #séparation du df en nb_groupes groupes
    d_split = split_val_croisee(df, nb_groupes)
    scores = []

    for group in range(nb_groupes):
        test = d_split[group]
        train = df.drop(test.index).reset_index(drop=True)
        train_x = train.values[:, :nb_features]
        train_y = train.values[:, -1]
        test_x = test.values[:, :nb_features]
        test_y = test.values[:, -1]
        clf.fit(train_x, train_y)
        predict = clf.predict(test_x)
        
        # précision
        taux_reussite = sum(predict == test_y) / len(predict)
        scores.append(taux_reussite)

    average = sum(scores) / len(scores)
    std = np.std(scores)

    return average, std

clf_ = LogisticRegression()
precision, robustesse = validation_croisee(data, nb_groupes=10, clf=clf_)
print("Précision (taux de réussite moyen) : {}\n"
      "Robustesse (écart-type du taux de réussite) : {}".format(precision, robustesse))

Précision (taux de réussite moyen) : 0.8686010436957412
Robustesse (écart-type du taux de réussite) : 0.01498243294192616


### Validation croisée imbriquée

On met ici en concurrence différents algorithmes et hyperparamètres.
On affiche quels modèles et paramètres ont été choisis à chaque itération de la boucle externe.

In [12]:
def validation_croisee_imbriquee(df, nb_groupes_internes, nb_groupes_externes):
    nb_features = len(df.columns) - 1

    # séparation du df en nb_groupes groupes
    df_split = split_val_croisee(df, nb_groupes_externes)
    out_averages = []
    best_model_params = []
    for index, df_test in df_split.items():
        in_averages = []
        #On construit le jeu de validation
        df_val = df.drop(df_test.index).reset_index(drop=True)

        for (model, params) in MODELS_HPP:
            clf = model(**params)
            average, std = validation_croisee(df_val, nb_groupes_internes, clf=clf)
            in_averages.append(average)
        
        # On trouve le meilleur modèle
        best_average_index = np.argmax(in_averages)
        best_model, best_params = MODELS_HPP[best_average_index]
        best_classifier = best_model(**best_params)
        
        # On entraîne le modèle sur toutes les données de validation et on prédit
        train_x = df_val.values[:, :nb_features]
        train_y = df_val.values[:, -1]
        test_x = df_test.values[:, :nb_features]
        test_y = df_test.values[:, -1]
        best_classifier.fit(train_x, train_y)
        predict = best_classifier.predict(test_x)
        
        # précision
        taux_reussite = sum(predict == test_y) / len(predict)
        out_averages.append(taux_reussite)
        best_model_params.append((best_model.__name__, best_params))

    average = sum(out_averages) / len(out_averages)
    std = np.std(out_averages)
    print("Best models used for average in nested cross validation : \n {}".format(best_model_params))
    return average, std

#On crée la liste pour le grid search.
MODELS = [MLPClassifier, SVC, DecisionTreeClassifier, LogisticRegression, GaussianNB]
PARAMETERS = [[{"hidden_layer_sizes" : d} for d in [(10,10), (20,20), (20,10), (10,20)]],
                  [{"kernel": "linear"}, {"kernel": "rbf"}, {"kernel": "sigmoid"}, {"kernel": "poly"}],
                  [{"criterion":"gini"}, {"criterion":"entropy"}],
                  [{"C" : c} for c in [0.01,1,100]],
                  [{}]
                  ]

MODELS_HPP = []
for i in range(len(MODELS)):
    MODELS_HPP.extend(list(itertools.product([MODELS[i]], PARAMETERS[i])))

precision, robustesse = validation_croisee_imbriquee(data, nb_groupes_internes=5, nb_groupes_externes=5)
print("Précision (taux de réussite moyen) : {}\n"
      "Robustesse (écart-type du taux de réussite) : {}".format(precision, robustesse))

Best models used for average in nested cross validation : 
 [('LogisticRegression', {'C': 100}), ('LogisticRegression', {'C': 100}), ('SVC', {'kernel': 'linear'}), ('LogisticRegression', {'C': 100}), ('LogisticRegression', {'C': 1})]
Précision (taux de réussite moyen) : 0.8687975951449005
Robustesse (écart-type du taux de réussite) : 0.007345170033382553


On voit que la régression logistique est utilisée le plus souvent, de plus avec le même hyperparamètre. Cela nous indique que ce modèle possède une précision au moins supérieure à celle des autres modèles (pour être sélectionné avant eux) et une forte robustesse (pour être selectionné plusieurs fois), c'est donc un comportement souhaitable.